# Southern Tree Species
Species
American Basswood
American Beech
American Elm
American sycamore 
Bald Cypress
Bigleaf Magnolia
Bitternut Hickory
Black Cherry
Black locust 
Black Oak 
Black walnut 
Black Willow
Blackgum 
Boxelder
Cherrybark Oak
Chestnut Oak
Chinkapin oak 
Common persimmon 
Eastern cottonwood 
Eastern Hophornbeam
Eastern Redbud
Eastern Redcedar
Eastern White Pine
Florida Maple
Flowering Dogwood
Green Ash
Hackberry
Honeylocust
Live Oak
Loblolly Pine
Longleaf Pine
Mimosa
Mockernut Hickory
Musclewood
Northern Red Oak
Ohio Buckeye
Osage-orange
Overcup Oak
Pecan
Pignut Hickory
Pin Oak 
Post Oak
Red Maple
River Birch
Sassafras
Sawtooth Oak
Scarlet Oak
Serviceberry
Shagbark hickory
shingle oak
Shortleaf Pine
silver maple
Slash Pine
slippery elm
Sourwood
Southern Magnolia
Southern Red Oak
sugar maple
Sugarberry
Swamp Chestnut Oak
Sweetbay Magnolia
sweetgum
Virginia Pine
Water Oak
White Ash
White Oak
Willow Oak
Winged Elm
Yellow Poplar

The following code block imports the labels CSV as a Pandas dataframe. It then displays the first five rows of the dataframe and prints the original row count.

The code filters out rows where the species was listed as "other" and prints the remaining row count.

Afterward, the Species column is selected as the target label, while the remaining columns are stored separately as input features. These inputs and labels are then split into training and testing sets. For initial exploration, only 5% of the data is used for training so that model runs stay fast

In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split

import os

IN_COLAB = os.path.exists("/content")

if IN_COLAB:
    import importlib

    if not os.path.exists("/content/drive/MyDrive"):
        drive = importlib.import_module("google.colab.drive")
        drive.mount("/content/drive")
    
    BASE_PATH = "/content/drive/MyDrive/Bark_Imagery/"
else:
    BASE_PATH = ""

labels_path = BASE_PATH + "bark_images_and_labels/bark_class_labels.csv"
extract_path = BASE_PATH + "bark_images_and_labels/BARK_UGA_PICS/"

df = pd.read_csv(labels_path)
display(df.head())
print("Original dataframe:", df.shape[0])

df = df[~df['Species'].str.lower().eq('other')].copy()
print("Dataframe after removing 'other' species:", df.shape[0])

print("Number of unique species:", df['Species'].nunique())

sample_tree_df, remaining_tree_df = train_test_split(
    df, 
    train_size=0.05, 
    random_state=42,
    stratify=df["Species"]
)
print("Training set size:", sample_tree_df.shape[0])
print("Number of unique species in training set:", sample_tree_df['Species'].nunique())

print("Remaining set size:", remaining_tree_df.shape[0])
print("Number of unique species in remaining set:", remaining_tree_df['Species'].nunique())

,OBJECTID,Species,Moisture_Condition,DBH_in,Note,CreatedBy,CreateTime,EditedBy,EditTime,Photo0,Photo1,Photo2,Photo3
0,1,Pin oak,Dry,17.0,NaN,tdb96463_USG,2024/02/05 18:42:18.856+00,tdb96463_USG,2024/02/05 18:42:18.856+00,ATT1_Photo 3.jpg,ATT2_Photo 2.jpg,ATT3_Photo 4.jpg,ATT4_Photo 1.jpg
1,2,Eastern white pine,Dry,29.0,NaN,tdb96463_USG,2024/02/05 18:49:21.604+00,tdb96463_USG,2024/02/05 18:49:21.604+00,ATT5_Photo 3.jpg,ATT6_Photo 2.jpg,ATT7_Photo 4.jpg,ATT8_Photo 1.jpg
2,3,Yellow-poplar,Dry,18.0,NaN,tdb96463_USG,2024/02/05 18:50:52.876+00,tdb96463_USG,2024/02/05 18:50:52.876+00,ATT9_Photo 2.jpg,ATT10_Photo 3.jpg,ATT11_Photo 1.jpg,ATT12_Photo 4.jpg
3,4,Loblolly pine,Dry,16.0,NaN,tdb96463_USG,2024/03/08 20:53:15.938+00,tdb96463_USG,2024/03/08 20:53:15.938+00,ATT13_Photo 2.jpg,ATT14_Photo 3.jpg,ATT15_Photo 4.jpg,ATT16_Photo 1.jpg
4,5,Loblolly pine,Dry,17.0,NaN,tdb96463_USG,2024/03/08 20:56:20.335+00,tdb96463_USG,2024/03/08 20:56:20.335+00,ATT17_Photo 1.jpg,ATT18_Photo 4.jpg,ATT19_Photo 3.jpg,ATT20_Photo 2.jpg


Original dataframe: 692
Dataframe after removing 'other' species: 685
Number of unique species: 25
Training set size: 34
Number of unique species in training set: 21
Remaining set size: 651
Number of unique species in remaining set: 25


In [43]:
species_counts_df = (
    df["Species"]
    .value_counts()
    .rename_axis("Species")
    .reset_index(name="TreeCount")
)

display(species_counts_df)

print("Number of species:", species_counts_df.shape[0])
print("Total trees:", species_counts_df["TreeCount"].sum())

,Species,TreeCount
0,Loblolly pine,172
1,Southern red oak,54
2,Willow oak,53
3,Live oak,50
4,Water oak,35
5,Scarlet oak,31
6,Swamp chestnut oak,29
7,Mockernut hickory,27
8,Bald cypress,22
9,Pecan,22


Number of species: 25
Total trees: 685


The following code block melts the sampled tree-level dataframe across the four photo columns. This converts the data from one row per tree, with separate Photo0, Photo1, Photo2, and Photo3 columns, into one row per image with a single Image column.

The OBJECTID and Species columns are preserved so that each melted image row still keeps its tree identifier and species label. Rows with missing image filenames are removed, and the index is reset.

The code then builds a full ImagePath column by combining the extracted image folder path with each filename in the Image column. It verifies that all images for tree 598 remain together in the sampled dataframe, displays the sample image count and first few rows, and checks whether the generated image paths exist on disk.

In [44]:
import os

photo_labels = ["Photo0", "Photo1", "Photo2", "Photo3"]

def melt_photos(split_df):
    melted_df = split_df.melt(
        id_vars=['OBJECTID', 'Species'], 
        value_vars=photo_labels, 
        var_name='Photo', 
        value_name='Image'
    )
    
    return melted_df.dropna(subset=['Image']).reset_index(drop=True)

def build_image_paths(image_df, extract_path):
    image_df['ImagePath'] = extract_path + image_df['Image']
    return image_df

sample_image_df = melt_photos(sample_tree_df)
sample_image_df = build_image_paths(sample_image_df, extract_path)
path_exists_df = sample_image_df["ImagePath"].apply(os.path.exists)

print("Verification of tree level split using tree number 598")
display(sample_image_df[sample_image_df["OBJECTID"] == 598])

print("Sample set image count:", sample_image_df.shape[0])
display(sample_image_df.head())

display(path_exists_df.head())
print(path_exists_df.value_counts())

Verification of tree level split using tree number 598


,OBJECTID,Species,Photo,Image,ImagePath
3,598,Water oak,Photo0,ATT2385_Photo 2.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
37,598,Water oak,Photo1,ATT2386_Photo 4.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
71,598,Water oak,Photo2,ATT2387_Photo 1.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
105,598,Water oak,Photo3,ATT2388_Photo 3.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...


Sample set image count: 136


,OBJECTID,Species,Photo,Image,ImagePath
0,40,Southern magnolia,Photo0,ATT157_Photo 4.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
1,54,Longleaf pine,Photo0,ATT213_Photo 2.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
2,615,Eastern white pine,Photo0,ATT2453_Photo 4.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
3,598,Water oak,Photo0,ATT2385_Photo 2.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
4,203,Willow oak,Photo0,ATT809_Photo 3.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...


,ImagePath
0,True
1,True
2,True
3,True
4,True


ImagePath
True    136
Name: count, dtype: int64


The following code block defines the folder path where the extracted bark images are stored. It then defines a function that creates a new ImagePath column by combining the base image folder path with each image filename from the Image column.

The function is applied to both the training and testing image dataframes. Finally, the first few rows of the training dataframe are displayed to verify that the image paths were created correctly.

In [45]:
from sklearn.metrics import accuracy_score, f1_score
import torch
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import gc
import numpy as np
from sklearn.model_selection import StratifiedKFold

class BarkDataset(Dataset):
        def __init__(self, image_df, transform=None):
            self.image_df = image_df
            self.transform = transform
        
        def __len__(self):
            return len(self.image_df)
        
        def __getitem__(self, index):
            row = self.image_df.iloc[index]
            image_path = row["ImagePath"]
            label = row["Label"]

            try:
                image = Image.open(image_path).convert("RGB")
            except Exception as e:
                raise RuntimeError(f"Error loading image at {image_path}") from e

            
            if self.transform is not None:
                image = self.transform(image)

            label = torch.tensor(label, dtype=torch.long)
            
            return image, label

class TrainTestBarkModel():
    """
        class is initialized with default values, image_df or tree_df must be provided, but not both. 
        If image_df is provided, it will be used directly for training. 
        If tree_df is provided, it will be split into train, val, and test sets, and then melted to create the image_df for each set. 
        The label encoding is done on the entire dataset to ensure consistent mapping across splits. 
        The melt_labels and extract_path can be customized if needed.
    """
    def __init__(
            self, 
            train_image_df=None,
            val_image_df=None,
            test_image_df=None,
            tree_df=None,
            melt_labels=["Photo0", "Photo1", "Photo2", "Photo3"],
            extract_path = 'bark_images_and_labels/BARK_UGA_PICS/',
            five_fold_cross_validation=False,
            data_fraction=1.0,
            train_size=0.7,
            val_size=0.1,
            test_size=0.2,
            random_state=42
        ):

        self.train_image_df = train_image_df
        self.val_image_df = val_image_df
        self.test_image_df = test_image_df

        if (train_image_df is None and tree_df is None) or (
            tree_df is not None and (
                train_image_df is not None 
                or val_image_df is not None 
                or test_image_df is not None
            )
        ):
            raise ValueError(
                "Either train_image_df or tree_df must be provided, but not both.\n"
                "val_image_df and test_image_df are optional but should only be provided with train_image_df."
            )
        elif train_image_df is not None:
            self.train_image_df = train_image_df.reset_index(drop=True).copy()

            self.set_label_info(self.train_image_df)
            
            self.train_image_df['Label'] = self.label_encoder.fit_transform(self.train_image_df['Species'])

            if self.val_image_df is not None:
                self.val_image_df = self.val_image_df.reset_index(drop=True).copy()
                self.val_image_df["Label"] = self.label_encoder.transform(self.val_image_df["Species"])

            if self.test_image_df is not None:
                self.test_image_df = self.test_image_df.reset_index(drop=True).copy()
                self.test_image_df["Label"] = self.label_encoder.transform(self.test_image_df["Species"])
        elif tree_df is not None:
            self.tree_df = tree_df.reset_index(drop=True).copy()
            self.melt_labels = melt_labels.copy()
            self.extract_path = extract_path
            self.data_fraction = data_fraction
            self.train_size = train_size
            self.val_size = val_size
            self.test_size = test_size
            self.random_state = random_state

            self.set_label_info(self.tree_df)

            if five_fold_cross_validation:
                self.cv_folds = self.split_data_five_fold(self.tree_df)
            else:
                self.train_image_df, self.val_image_df, self.test_image_df = self.split_data(self.tree_df)

    """
        Sets the number of classes.
        Creates a label map dataframe based on the provided image_df.
    """
    def set_label_info(self, label_fit_df):
        label_encoder = LabelEncoder()
        label_encoder.fit(label_fit_df["Species"])

        self.num_classes = len(label_encoder.classes_)

        self.label_map_df = pd.DataFrame({
            "Label": range(len(label_encoder.classes_)),
            "Species": label_encoder.classes_
        })

        self.label_encoder = label_encoder

    """
        This def is called by the class initializer if a tree_df is provided. It performs the following steps:
        1. Validates the data_fraction and split sizes.
        2. If data_fraction is less than 1.0, it randomly samples the specified fraction of the tree_df while maintaining the class distribution.
        3. Splits the selected tree_df into train and temp sets based on the specified train_size, maintaining class distribution.
        4. Calculates the relative size for the test set and splits the temp set into val and test sets, maintaining class distribution.
        5. Fits a label encoder on the entire tree_df to ensure consistent mapping across splits and applies the encoding to the train, val, and test sets.
        6. Melts the photo columns for each split to create the image_df for train, val, and test sets.
        7. Builds the image paths for each split by concatenating the extract_path with the image filenames.
        8. Returns the train_image_df, val_image_df, and test_image_df.
    """
    def split_data(self, tree_df):
        if not (0.0 < self.data_fraction <= 1.0):
            raise ValueError("Data fraction must be between 0.0 and 1.0")
        
        if not np.isclose(self.train_size + self.val_size + self.test_size, 1.0):
            raise ValueError("Train, validation, and test sizes must sum to 1.0")
        
        # Based on the requested data_fraction, train_size, val_size, and test_size, does each species probably have enough trees before trying to split?
        temp_fraction = self.data_fraction * (self.val_size + self.test_size)

        self.validate_min_species_count(
            tree_df,
            required_fraction=temp_fraction,
            context="this stratified train/validation/test split"
        )

        if self.data_fraction < 1.0:
            selected_tree_df, _ = train_test_split(
                tree_df,
                train_size=self.data_fraction,
                random_state=self.random_state,
                stratify=tree_df["Species"]
            )
        else:
            selected_tree_df = tree_df.copy()

        train_tree_df, temp_tree_df = train_test_split(
            selected_tree_df,
            train_size=self.train_size,
            random_state=self.random_state,
            stratify=selected_tree_df["Species"]
        )

        # After the initial train split, check if the temp_tree_df has at least 2 trees per species before trying to split into val and test
        temp_species_counts = temp_tree_df["Species"].value_counts()
        too_small_temp_species = temp_species_counts[temp_species_counts < 2]

        if len(too_small_temp_species) > 0:
            raise ValueError(
                "Some species have fewer than 2 trees in the validation/test pool after splitting.\n"
                "Increase data_fraction, lower train_size, or remove rare species.\n\n"
                f"Too-small temp species:\n{too_small_temp_species}"
            )

        # Calculate the relative size of the test set with respect to the temp set to ensure the correct proportions for val and test splits
        test_relative_size = self.test_size / (self.val_size + self.test_size)

        val_tree_df, test_tree_df = train_test_split(
            temp_tree_df,
            test_size=test_relative_size,
            random_state=self.random_state,
            stratify=temp_tree_df["Species"]
        )

        train_image_df, val_image_df, test_image_df = self.build_image_splits(
            train_tree_df, 
            val_tree_df, 
            test_tree_df
        )
        
        return train_image_df, val_image_df, test_image_df

    def split_data_five_fold(self, tree_df):
        if not np.isclose(self.train_size + self.val_size, 0.8):
            raise ValueError("Train and validation sizes must sum to 0.8")
    
        if self.data_fraction != 1.0:
            raise ValueError(
                "data_fraction is not supported for five-fold cross-validation. "
                "Use data_fraction=1.0 so every eligible tree appears in exactly one test fold."
            )
        
        if not np.isclose(self.test_size, 0.2):
            raise ValueError(
                "five_fold_cross_validation=True requires test_size=0.2, "
                "because five folds means each test fold is 20% of the data."
            )
        
        n_splits = 5

        train_val_fraction = (n_splits - 1) / n_splits
        val_relative_size = self.val_size / (self.train_size + self.val_size)
        validation_fraction = train_val_fraction * val_relative_size

        self.validate_min_species_count(
            tree_df,
            required_fraction=validation_fraction,
            context="five-fold cross-validation with an inner validation split"
        )
        
        skf = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=self.random_state
        )

        cv_folds = []

        for fold_number, (train_val_idx, test_idx) in enumerate(
            skf.split(tree_df, tree_df["Species"]),
            start=1
        ):
            train_val_tree_df = tree_df.iloc[train_val_idx].copy()
            test_tree_df = tree_df.iloc[test_idx].copy()

            val_relative_size = self.val_size / (self.train_size + self.val_size)

            train_tree_df, val_tree_df = train_test_split(
                train_val_tree_df,
                test_size=val_relative_size,
                random_state=self.random_state + fold_number,
                stratify=train_val_tree_df["Species"]
            )

            train_image_df, val_image_df, test_image_df = self.build_image_splits(
                train_tree_df,
                val_tree_df,
                test_tree_df
            )

            cv_folds.append({
                "fold": fold_number,
                "train_tree_df": train_tree_df,
                "val_tree_df": val_tree_df,
                "test_tree_df": test_tree_df,
                "train_image_df": train_image_df,
                "val_image_df": val_image_df,
                "test_image_df": test_image_df,
            })

        return cv_folds
    
    def set_fold(self, fold_index):
        fold = self.cv_folds[fold_index]
        
        self.train_image_df = fold["train_image_df"]
        self.val_image_df = fold["val_image_df"]
        self.test_image_df = fold["test_image_df"]
    
    def build_image_splits(self, train_tree_df, val_tree_df, test_tree_df):
        #apply the same label encoding to all splits
        train_tree_df['Label'] = self.label_encoder.transform(train_tree_df['Species'])
        val_tree_df['Label'] = self.label_encoder.transform(val_tree_df['Species'])
        test_tree_df['Label'] = self.label_encoder.transform(test_tree_df['Species'])

        train_image_df = self.melt_photos(train_tree_df)
        val_image_df = self.melt_photos(val_tree_df)
        test_image_df = self.melt_photos(test_tree_df)

        train_image_df = self.build_image_paths(train_image_df)
        val_image_df = self.build_image_paths(val_image_df)
        test_image_df = self. build_image_paths(test_image_df)

        return train_image_df, val_image_df, test_image_df

    
    """
        Utility function to melt the photo columns into a long format dataframe with one row per image.
        The function takes a split_df (train, val, or test) and melts the specified photo columns into a 
        long format dataframe with columns for OBJECTID, Species, Photo, and Image.
    """
    def melt_photos(self, split_df):
        melted_df = split_df.melt(
            id_vars=['OBJECTID', 'Species', 'Label'], 
            value_vars=self.melt_labels, 
            var_name='Photo', 
            value_name='Image'
        )
        
        return melted_df.dropna(subset=['Image']).reset_index(drop=True)
    
    """
        Utility function to build the full image paths by concatenating the extract_path with the image filenames.
        The function takes an image_df with an 'Image' column containing the image filenames and adds a new 'ImagePath' 
        column with the full path to each image.
    """
    def build_image_paths(self, image_df):
        image_df['ImagePath'] = self.extract_path + image_df['Image']
        return image_df
    
    def validate_min_species_count(
        self,
        tree_df,
        required_fraction,
        context,
        min_examples=2,
    ):
        min_trees_needed = int(np.ceil(min_examples / required_fraction))

        species_counts = tree_df["Species"].value_counts()
        too_small_species = species_counts[species_counts < min_trees_needed]

        if len(too_small_species) > 0:
            raise ValueError(
                f"Some species do not have enough trees for {context}.\n"
                f"Each species needs at least {min_trees_needed} trees.\n\n"
                f"Too-small species:\n{too_small_species}"
            )

        return min_trees_needed

    """
        Called by user.
        Displays the label to species mapping and the number of classes.
    """
    def show_label_map(self):
        print("Label to Species mapping: ", self.num_classes, " classes")
        display(self.label_map_df)

    def get_num_classes(self):
        return self.num_classes
        
    def get_num_images(self, image_df=None):
        if image_df is None:
            raise ValueError("image_df must be provided to get the number of images.")
        return len(image_df)

    def get_dataloader(self, batch_size, shuffle, image_df=None, transform=None):
        if image_df is None:
            image_df = self.train_image_df

        if transform is None:
            raise ValueError("transform must be provided for training/evaluation dataloaders.")

        dataset = BarkDataset(
            image_df,
            transform=transform
        )

        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=2,
            pin_memory=torch.cuda.is_available()
        )

        return dataloader
    
    def get_transform(self, weights=None, image_size=None):
        if image_size is None and weights is not None:
            return weights.transforms()

        if image_size is None:
            image_size = (224, 224)

        return transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            )
        ])
    
    def iter_training_splits(self):
        if hasattr(self, "cv_folds") and self.cv_folds is not None:
            for fold in self.cv_folds:
                yield {
                    "fold": fold["fold"],
                    "train_image_df": fold["train_image_df"],
                    "val_image_df": fold["val_image_df"],
                    "test_image_df": fold["test_image_df"],
                }
        else:
            yield {
                "fold": None,
                "train_image_df": self.train_image_df,
                "val_image_df": self.val_image_df,
                "test_image_df": self.test_image_df,
            }
    
    def train_model(self, model_tuple, batch_size, num_epochs):
        model_name, model_fn, weights, image_size = model_tuple

        results = []

        for split_context in self.iter_training_splits():
            fold = split_context["fold"]
            train_image_df = split_context["train_image_df"]
            val_image_df = split_context["val_image_df"]
            test_image_df = split_context["test_image_df"]

            self.train_image_df = train_image_df
            self.val_image_df = val_image_df
            self.test_image_df = test_image_df

            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

            current_num_epochs = num_epochs
            if current_num_epochs is None:
                current_num_epochs = self.get_num_images(self.train_image_df) // batch_size

            model = model_fn(weights=weights)

            train_transform = self.get_transform(weights=weights, image_size=image_size)
            eval_transform = self.get_transform(weights=weights, image_size=image_size)

            if hasattr(model, "fc"):
                model.fc = nn.Linear(model.fc.in_features, self.num_classes)
            elif hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
                model.classifier[-1] = nn.Linear(
                    model.classifier[-1].in_features,
                    self.num_classes
                )
            else:
                raise ValueError("Unsupported model architecture")

            model = model.to(device)

            fold_label = f" experiment fold {fold}" if fold is not None else ""
            print(f"\n===== Training {model_name}{fold_label} =====")
            print(f"Model loaded on device: {device}")
            print(f"Number of classes: {self.num_classes}")
            print()

            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

            # Training dataloader
            train_dataloader = self.get_dataloader(
                batch_size=batch_size,
                shuffle=True,
                image_df=self.train_image_df,
                transform=train_transform
            )

            # Validation and testing dataloader
            if self.val_image_df is not None:
                val_test_dataloader = self.get_dataloader(
                    batch_size=batch_size,
                    shuffle=False,
                    image_df=self.val_image_df,
                    transform=eval_transform
                )

            if self.test_image_df is not None:
                test_dataloader = self.get_dataloader(
                    batch_size=batch_size,
                    shuffle=False,
                    image_df=self.test_image_df,
                    transform=eval_transform
                )

            best_val_macro_f1 = -1.0
            best_epoch = 0

            for epoch in range(current_num_epochs):
                epoch_loss, epoch_accuracy, epoch_macro_f1 = self.train_one_epoch(
                    model,
                    train_dataloader,
                    criterion,
                    optimizer,
                    device
                )

                # (self, model, dataloader, criterion, split_name)
                if self.val_image_df is not None:
                    val_loss, val_acc, val_macro_f1 = self.evaluate_model(
                        model,
                        val_test_dataloader,
                        criterion,
                        "Validation"
                    )

                print(f"Epoch {epoch + 1}/{current_num_epochs}")
                print(f"TrainingLoss: {epoch_loss:.4f}")
                print(f"Training Accuracy: {epoch_accuracy:.4f}")
                print(f"Training Macro F1: {epoch_macro_f1:.4f}")
                print(f"Validation Loss: {val_loss:.4f}")
                print(f"Validation Accuracy: {val_acc:.4f}")
                print(f"Validation Macro F1: {val_macro_f1:.4f}")

                if IN_COLAB:
                    import importlib

                    if not os.path.exists("/content/drive/MyDrive"):
                        drive = importlib.import_module("google.colab.drive")
                        drive.mount("/content/drive")
                    
                    output_dir = "/content/drive/MyDrive/bark_model_runs"
                else:
                    output_dir = "./bark_model_runs"

                os.makedirs(output_dir, exist_ok=True)

                weights_label = "_imagenet" if weights is not None else "random"
                fold_label_for_file = f"_fold{fold}" if fold is not None else ""
                
                checkpoint_path = os.path.join(
                    output_dir,
                    f"{model_name}{weights_label}{fold_label_for_file}_best.pth"
                )


                if val_macro_f1 > best_val_macro_f1:
                    best_val_macro_f1 = val_macro_f1
                    best_epoch = epoch + 1

                    torch.save(
                        {
                            "model_name": model_name,
                            "weights": str(weights) if weights is not None else None,
                            "image_size": image_size,
                            "num_classes": self.num_classes,
                            "class_names": self.label_encoder.classes_.tolist(),

                            "epoch": best_epoch,
                            "best_val_macro_f1": float(best_val_macro_f1),

                            "model_state_dict": {
                                k: v.detach().cpu()
                                for k, v in model.state_dict().items()
                            },
                        },
                        checkpoint_path
                    )

                    print(f"Saved new best checkpoint: {checkpoint_path}\n")
                else:
                    print()

            test_loss, test_acc, test_macro_f1 = None, None, None

            if self.test_image_df is not None:
                checkpoint = torch.load(checkpoint_path, map_location=device)
                model.load_state_dict(checkpoint["model_state_dict"])

                test_loss, test_acc, test_macro_f1 = self.evaluate_model(
                    model,
                    test_dataloader,
                    criterion,
                    "Test"
                )

            print(f"Test Loss: {test_loss:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(f"Test Macro F1: {test_macro_f1:.4f}")

            results.append({
                "model": model_name,
                "fold": fold,
                "status": "passed",
                "epochs": current_num_epochs,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "test_loss": test_loss,
                "test_accuracy": test_acc,
                "error": None,
            })

            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        results_df = pd.DataFrame(results)
        display(results_df)

        return results_df
    
    def train_one_epoch(self, model, dataloader, criterion, optimizer, device):
        model.train()

        running_loss = 0.0
        all_predictions = []
        all_labels = []

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            predicted_labels = outputs.argmax(dim=1)
            
            all_predictions.extend(predicted_labels.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_accuracy = accuracy_score(all_labels, all_predictions)

        epoch_macro_f1 = f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )

        return epoch_loss, epoch_accuracy, epoch_macro_f1

    def evaluate_model(self, model, dataloader, criterion, split_name):
        device = next(model.parameters()).device
        model.eval()

        running_loss = 0.0
        all_predictions = []
        all_labels = []

        with torch.no_grad():
            for images, labels in dataloader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                predicted_labels = outputs.argmax(dim=1)

                all_predictions.extend(predicted_labels.detach().cpu().numpy())
                all_labels.extend(labels.detach().cpu().numpy())

        if len(all_labels) == 0:
            raise ValueError(f"{split_name} split has no images.")
        
        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_accuracy = accuracy_score(all_labels, all_predictions)

        val_macro_f1 = f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )

        return epoch_loss, epoch_accuracy, val_macro_f1


    def train_multiple(self, models, batch_size, num_epochs):
        all_results = []

        for model_tuple in models:
            model_name = model_tuple[0]

            try:
                model_results_df = self.train_model(
                    model_tuple=model_tuple,
                    batch_size=batch_size,
                    num_epochs=num_epochs,
                )

                if model_results_df is None:
                    model_results_df = pd.DataFrame([{
                        "model": model_name,
                        "status": "passed",
                        "error": None,
                    }])

                all_results.append(model_results_df)

            except KeyboardInterrupt:
                print(f"Training interrupted during {model_name}.")
                raise

            except Exception as e:
                print(f"Training failed for {model_name}: {e}")

                all_results.append(pd.DataFrame([{
                    "model": model_name,
                    "status": "failed",
                    "error": str(e),
                }]))

            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        results_df = pd.concat(all_results, ignore_index=True)
        display(results_df)

        return results_df

In [46]:
min_trees_per_species = 34

species_counts = df["Species"].value_counts()
kept_species = species_counts[species_counts >= min_trees_per_species].index

df_filtered_species = df[df["Species"].isin(kept_species)].copy()

print("Original species:", df["Species"].nunique())
print("Kept species:", df_filtered_species["Species"].nunique())
print("Removed species:")
display(species_counts[species_counts < min_trees_per_species])

display(
    df_filtered_species["Species"]
    .value_counts()
    .rename_axis("Species")
    .reset_index(name="TreeCount")
)

Original species: 25
Kept species: 5
Removed species:


,count
Species,
Scarlet oak,31
Swamp chestnut oak,29
Mockernut hickory,27
Bald cypress,22
Pecan,22
River birch,21
Sourwood,21
Eastern white pine,19
Shortleaf pine,18


,Species,TreeCount
0,Loblolly pine,172
1,Southern red oak,54
2,Willow oak,53
3,Live oak,50
4,Water oak,35


In [47]:
from torchvision.models import (
    # ResNet models
    resnet18,
    resnet34,
    resnet50,
    resnet101,
    resnet152,

    # EfficientNet B models
    efficientnet_b0,
    efficientnet_b1,
    efficientnet_b2,
    efficientnet_b3,
    efficientnet_b4,
    efficientnet_b5,
    efficientnet_b6,
    efficientnet_b7,

    # EfficientNet V2 models
    efficientnet_v2_s,
    efficientnet_v2_m,
    efficientnet_v2_l,

    ResNet18_Weights,
    ResNet34_Weights,
    ResNet50_Weights,
    ResNet101_Weights,
    ResNet152_Weights,

    EfficientNet_B0_Weights,
    EfficientNet_B1_Weights,
    EfficientNet_B2_Weights,
    EfficientNet_B3_Weights,
    EfficientNet_B4_Weights,
    EfficientNet_B5_Weights,
    EfficientNet_B6_Weights,
    EfficientNet_B7_Weights,

    EfficientNet_V2_S_Weights,
    EfficientNet_V2_M_Weights,
    EfficientNet_V2_L_Weights   
)

models_default_weights_custom_imagesize_full = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, (224, 224)),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, (224, 224)),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, (224, 224)),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, (224, 224)),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, (224, 224)),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, (224, 224)),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, (240, 240)),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, (288, 288)),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, (300, 300)),
    ("efficientnet_b4", efficientnet_b4, EfficientNet_B4_Weights.DEFAULT, (380, 380)),
    ("efficientnet_b5", efficientnet_b5, EfficientNet_B5_Weights.DEFAULT, (456, 456)),
    ("efficientnet_b6", efficientnet_b6, EfficientNet_B6_Weights.DEFAULT, (528, 528)),
    ("efficientnet_b7", efficientnet_b7, EfficientNet_B7_Weights.DEFAULT, (600, 600)),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, (384, 384)),
    ("efficientnet_v2_m", efficientnet_v2_m, EfficientNet_V2_M_Weights.DEFAULT, (480, 480)),
    ("efficientnet_v2_l", efficientnet_v2_l, EfficientNet_V2_L_Weights.DEFAULT, (480, 480)),
]

models_default_weights_full = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, None),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, None),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, None),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, None),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, None),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, None),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, None),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, None),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, None),
    ("efficientnet_b4", efficientnet_b4, EfficientNet_B4_Weights.DEFAULT, None),
    ("efficientnet_b5", efficientnet_b5, EfficientNet_B5_Weights.DEFAULT, None),
    ("efficientnet_b6", efficientnet_b6, EfficientNet_B6_Weights.DEFAULT, None),
    ("efficientnet_b7", efficientnet_b7, EfficientNet_B7_Weights.DEFAULT, None),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, None),
    ("efficientnet_v2_m", efficientnet_v2_m, EfficientNet_V2_M_Weights.DEFAULT, None),
    ("efficientnet_v2_l", efficientnet_v2_l, EfficientNet_V2_L_Weights.DEFAULT, None),
]

models_default_weights_custom_imagesize_partial = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, (224, 224)),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, (224, 224)),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, (224, 224)),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, (224, 224)),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, (224, 224)),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, (224, 224)),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, (240, 240)),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, (288, 288)),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, (300, 300)),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, None),
]

models_default_weights_partial = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, None),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, None),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, None),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, None),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, None),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, None),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, None),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, None),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, None),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, None),
]

In [ ]:
import contextlib
import threading
import traceback

import ipywidgets as widgets
from IPython.display import display, clear_output

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

BATCH_SIZE = globals().get("BATCH_SIZE", 32)
NUM_EPOCHS = globals().get("NUM_EPOCHS", None)

model_registry = {
    model_name: model_tuple
    for model_tuple in models_default_weights_full
    for model_name in [model_tuple[0]]
}

model_select_label = widgets.HTML(
    value="<b>Models:</b> select one or more (hold Ctrl or Cmd to select multiple)"
)

model_select = widgets.SelectMultiple(
    options=list(model_registry.keys()),
    value=("resnet18",),
    description="",
    rows=10,
    layout=widgets.Layout(width="420px"),
)

data_mode_dropdown = widgets.Dropdown(
    options=[
        ("Image dataframe", "image_df"),
        ("Tree dataframe with custom split", "tree_split"),
        ("5-fold validation", "five_fold"),
    ],
    value="five_fold",
    description="Data:",
    layout=widgets.Layout(width="420px"),
)

batch_size_input = widgets.IntText(
    value=BATCH_SIZE,
    description="Batch size:",
    layout=widgets.Layout(width="220px"),
)

num_epochs_input = widgets.Text(
    value="None" if NUM_EPOCHS is None else str(NUM_EPOCHS),
    description="Epochs:",
    layout=widgets.Layout(width="220px"),
)

run_button = widgets.Button(
    description="Run training",
    button_style="success",
)

cancel_button = widgets.Button(
    description="Cancel",
    button_style="warning",
    disabled=True,
)

status_label = widgets.HTML(value="")

gui_output = widgets.Output(
    layout=widgets.Layout(
        width="840px",
        max_height="320px",
        overflow="auto",
        border="1px solid #333",
    )
)

cancel_training = threading.Event()
training_thread = None

class WidgetStream:
    def __init__(self, append_fn):
        self.append_fn = append_fn

    def write(self, text):
        if text:
            self.append_fn(text)

    def flush(self):
        pass

def parse_num_epochs(value):
    value = value.strip()
    if value.lower() in ("", "none"):
        return None
    return int(value)

def make_cancelable(trainer, cancel_event):
    def raise_if_cancelled():
        if cancel_event.is_set():
            raise KeyboardInterrupt("Training cancelled by user.")

    def cancelable_train_one_epoch(model, dataloader, criterion, optimizer, device):
        model.train()

        running_loss = 0.0
        all_predictions = []
        all_labels = []

        for images, labels in dataloader:
            raise_if_cancelled()
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            predicted_labels = outputs.argmax(dim=1)

            all_predictions.extend(predicted_labels.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

        raise_if_cancelled()
        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_accuracy = accuracy_score(all_labels, all_predictions)
        epoch_macro_f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)

        return epoch_loss, epoch_accuracy, epoch_macro_f1

    def cancelable_evaluate_model(model, dataloader, criterion, split_name):
        device = next(model.parameters()).device
        model.eval()

        running_loss = 0.0
        all_predictions = []
        all_labels = []

        with torch.no_grad():
            for images, labels in dataloader:
                raise_if_cancelled()
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                predicted_labels = outputs.argmax(dim=1)

                all_predictions.extend(predicted_labels.detach().cpu().numpy())
                all_labels.extend(labels.detach().cpu().numpy())

        raise_if_cancelled()
        if len(all_labels) == 0:
            raise ValueError(f"{split_name} split has no images.")

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_accuracy = accuracy_score(all_labels, all_predictions)
        val_macro_f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)

        return epoch_loss, epoch_accuracy, val_macro_f1

    trainer.train_one_epoch = cancelable_train_one_epoch
    trainer.evaluate_model = cancelable_evaluate_model
    return trainer

def run_custom(selected_model_names, data_mode, batch_size, num_epochs, cancel_event):
    selected_models = [model_registry[model_name] for model_name in selected_model_names]

    if len(selected_models) == 0:
        raise ValueError("Select at least one model.")

    if data_mode == "image_df":
        trainer = TrainTestBarkModel(sample_image_df, extract_path=extract_path)
    elif data_mode == "tree_split":
        trainer = TrainTestBarkModel(tree_df=df_filtered_species, extract_path=extract_path, data_fraction=0.2)
    elif data_mode == "five_fold":
        trainer = TrainTestBarkModel(tree_df=df_filtered_species, extract_path=extract_path, five_fold_cross_validation=True)
    else:
        raise ValueError(f"Unsupported data mode: {data_mode}")

    trainer = make_cancelable(trainer, cancel_event)
    if len(selected_models) == 1:
        return trainer.train_model(model_tuple=selected_models[0], batch_size=batch_size, num_epochs=num_epochs)

    return trainer.train_multiple(models=selected_models, batch_size=batch_size, num_epochs=num_epochs)

def set_training_state(is_training):
    run_button.disabled = is_training
    cancel_button.disabled = not is_training
    run_button.description = "Training..." if is_training else "Run training"
    run_button.button_style = "" if is_training else "success"
    status_label.value = "<b>Training models...</b>" if is_training else ""

def on_cancel_button_clicked(_):
    cancel_training.set()
    cancel_button.disabled = True
    status_label.value = "<b>Cancel requested...</b>"
    gui_output.append_stdout("\nCancel requested; stopping after the current batch.\n")

def training_worker(selected_model_names, data_mode, batch_size, num_epochs):
    stdout_stream = WidgetStream(gui_output.append_stdout)
    stderr_stream = WidgetStream(gui_output.append_stderr)
    original_display = globals().get("display")

    def widget_display(*objects, **kwargs):
        for obj in objects:
            gui_output.append_display_data(obj)

    try:
        globals()["display"] = widget_display
        with contextlib.redirect_stdout(stdout_stream), contextlib.redirect_stderr(stderr_stream):
            print("Running custom training selection...", flush=True)
            run_custom(selected_model_names, data_mode, batch_size, num_epochs, cancel_training)
            print("Training complete.", flush=True)
    except KeyboardInterrupt:
        gui_output.append_stdout("\nTraining cancelled.\n")
    except Exception:
        gui_output.append_stderr(traceback.format_exc())
    finally:
        if original_display is not None:
            globals()["display"] = original_display
        set_training_state(False)

def on_run_button_clicked(_):
    global training_thread

    if training_thread is not None and training_thread.is_alive():
        return

    gui_output.clear_output()
    cancel_training.clear()
    set_training_state(True)
    try:
        batch_size = batch_size_input.value
        num_epochs = parse_num_epochs(num_epochs_input.value)
        selected_model_names = tuple(model_select.value)
        data_mode = data_mode_dropdown.value
    except Exception:
        set_training_state(False)
        gui_output.append_stderr(traceback.format_exc())
        return

    training_thread = threading.Thread(
        target=training_worker,
        args=(selected_model_names, data_mode, batch_size, num_epochs),
        daemon=True,
    )
    training_thread.start()

run_button.on_click(on_run_button_clicked)
cancel_button.on_click(on_cancel_button_clicked)

gui_css = widgets.HTML("""
<style>
.bark-training-gui {
    background: #000;
    color: #fff;
    padding: 16px;
    border-radius: 8px;
    border: 1px solid #333;
}
.bark-training-gui label,
.bark-training-gui .widget-label,
.bark-training-gui .widget-html-content {
    color: #fff !important;
}
.bark-training-gui select,
.bark-training-gui input,
.bark-training-gui textarea {
    background: #111 !important;
    color: #fff !important;
    border: 1px solid #444 !important;
}
.bark-training-gui option {
    background: #111;
    color: #fff;
}
.bark-training-gui .jupyter-widgets-output-area {
    background: #050505;
    color: #fff !important;
    min-height: 320px;
}
.bark-training-gui .jupyter-widgets-output-area pre,
.bark-training-gui .output,
.bark-training-gui .output_text,
.bark-training-gui .output_stream,
.bark-training-gui .widget-output {
    background: #050505 !important;
    color: #fff !important;
    font-family: Menlo, Consolas, monospace;
    font-size: 13px;
    white-space: pre-wrap;
}
</style>
""")

gui = widgets.VBox([
    model_select_label,
    model_select,
    data_mode_dropdown,
    widgets.HBox([batch_size_input, num_epochs_input]),
    widgets.HBox([run_button, cancel_button, status_label]),
    gui_output,
], layout=widgets.Layout(background_color="#000", padding="16px", border="1px solid #333"))
gui.add_class("bark-training-gui")

display(gui_css, gui)


HTML(value='\n<style>\n.bark-training-gui {\n    background: #000;\n    color: #fff;\n    padding: 16px;\n    …

In [49]:
# 0 for single model training
# 1 for full model training without evaluation
# 2 for full model training with evaluation using default weights
# 3 for partial model training with evaluation using default weights
# 4 for full model training with evaluation using default weights and custom image sizes
# 5 for partial model training with evaluation using default weights and custom image sizes
# 6 for five fold cross-validation on full model list with default weights
# 7 ------------- for five fold cross-validation on partial model list with default weights -------------
# 8 for five fold cross-validation on full model list with default weights and custom image sizes
# 9 for five fold cross-validation on partial model list with default weights and custom image sizes

MODEL_OPTION = None
BATCH_SIZE = 32
NUM_EPOCHS = None

bulk_trainer = TrainTestBarkModel(sample_image_df, extract_path=extract_path)
bulk_trainer_and_eval = TrainTestBarkModel(tree_df=df_filtered_species, extract_path=extract_path, data_fraction=0.2)
bulk_trainer_and_cross_val = TrainTestBarkModel(tree_df=df_filtered_species, extract_path=extract_path, five_fold_cross_validation=True)

if MODEL_OPTION == 0: # test if the training loop works with a single model before trying to train multiple models
    bulk_trainer.train_model(model_tuple=("resnet18", resnet18, None, (224, 224)), batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 1: # test which models can train without running out of memory
    bulk_trainer.train_multiple(models=models_default_weights_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 2: # train and eval on all models using default weights
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 3: # train and eval on partial model list with default weights
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_partial, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 4: # train and eval on all models using default weights and custom image sizes
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_custom_imagesize_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 5: # train and eval on partial model list with default weights and custom image sizes
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_custom_imagesize_partial, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 6: # five fold cross-validation on all models using default weights
    bulk_trainer_and_cross_val.train_multiple(models=models_default_weights_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 7: # five fold cross-validation on partial model list with default weights
    bulk_trainer_and_cross_val.train_multiple(models=models_default_weights_partial, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 8: # five fold cross-validation on all models using default weights and custom image sizes
    bulk_trainer_and_cross_val.train_multiple(models=models_default_weights_custom_imagesize_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 9: # five fold cross-validation on partial model list with default weights and custom image sizes
    bulk_trainer_and_cross_val.train_multiple(models=models_default_weights_custom_imagesize_partial, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)